# AI Exercises – Adversarial Search, Game Playing and Q-Learning

<i>
Jan Schlüter, Tara Jadidi, Tobias Morocutti, Patricia Hu, Alexander Fichtinger, Nikita Lvov
<br/>
Institute of Computational Perception, Johannes Kepler University Linz
</i>

This notebook walks you through implementing algorithms for game playing and reinforcement learning. In the first part, you will write **minimax** and **alpha-beta pruning** and use them to play tic-tac-toe (adversarial search). In the second part, you will build a controller for an **elevator**: with **Q-learning** from the lecture.

## If needed: Google Colab Setup

**Only if you're running this notebook on Google Colab**, run the next cell to fetch the other files it depends on (helper modules, maps, sudokus).

**If you're running locally or on Binder**, this is not needed, and it will do nothing. Just skip it.

In [ ]:
# If you're running this cell on Google Colab, it will copy in the remaining files.
# Otherwise, it doesn't do anything.

import os, sys

if 'google.colab' in sys.modules and not os.path.exists('search/tictactoe.py'):
    if not os.path.exists('ai-exercises'):
        !git clone --quiet --branch topic03 --depth 1 https://github.com/CPJKU/ai-exercises.git
    %cd ai-exercises

## Adversarial Search

In the search exercises, you chose every action yourself. In a game, there is also an
**opponent**, who chooses actions too and wants the opposite of what you want. **Minimax**
finds the best action, assuming that the opponent plays as well as possible. **Alpha-beta
pruning** finds the same action, but skips branches that cannot change the result.

### Tic-tac-toe

Our game is tic-tac-toe. `X` moves first, `O` moves second. The nine cells are numbered row
by row:

```
 0 | 1 | 2
---+---+---
 3 | 4 | 5
---+---+---
 6 | 7 | 8
```

A state is a string of nine characters, one for each cell: `X`, `O`, or `.` for an empty
cell. For example, `XO.......` has an `X` in cell 0 and an `O` in cell 1.

In [ ]:
import math

from search.tictactoe import TicTacToe, print_board

game = TicTacToe()
state = game.get_start()
print(repr(state))
print_board(state)

We call `X` the **MAX** player: it tries to make the result as large as possible. `O` is
the **MIN** player and tries to make it as small as possible. The game is already
implemented in `search/tictactoe.py`. You only need these methods:

| method | what it returns |
|---|---|
| `get_start()` | the empty board |
| `is_terminal(state)` | `True` if somebody has won or the board is full |
| `utility(state)` | for a finished game: `+1` if X has won, `-1` if O has won, `0` for a draw |
| `is_max_turn(state)` | `True` if it is X's turn |
| `result(state, action)` | the new state after the player to move has played the cell number `action` |
| `successors(state)` | a list of `(action, next_state)` pairs, one for every empty cell. The action is the number of that cell. |

The game also counts how often you called `successors()`, in `game.expansions`. We will use
that to compare the two algorithms.

In [ ]:
for action, next_state in game.successors(state)[:2]:
    print("action", action)
    print_board(next_state)

### Minimax

Minimax looks at the whole game tree below a state. At the bottom, the game is over, and
the utility says who won. In a state where MAX is to move, MAX picks the move with the
largest value. Where MIN is to move, MIN picks the move with the smallest value. So the
value of a state is the result of the game if both players play perfectly.

Here is the algorithm from the lecture:

<div style="font-family: 'CMU Serif', 'Latin Modern Roman', Georgia, 'Times New Roman', serif; font-size: 1.05em; line-height: 1.6; margin: 1em 0;"><b>function</b> <span style="font-variant: small-caps; color: #a31515;">Minimax-Decision</span>(<i>state</i>) <b>returns</b> an action
<div style="margin-left: 1.5em;">
<b>return</b> the <i>a</i> in <span style="font-variant: small-caps;">Actions</span>(<i>state</i>) with the highest <span style="font-variant: small-caps;">Min-Value</span>(<span style="font-variant: small-caps;">Result</span>(<i>state</i>, <i>a</i>))
</div>
<br>
<b>function</b> <span style="font-variant: small-caps; color: #a31515;">Max-Value</span>(<i>state</i>) <b>returns</b> a utility value
<div style="margin-left: 1.5em;">
<b>if</b> <span style="font-variant: small-caps;">Terminal-Test</span>(<i>state</i>) <b>then return</b> <span style="font-variant: small-caps;">Utility</span>(<i>state</i>)<br>
<i>v</i> &larr; &minus;&infin;<br>
<b>for each</b> <i>a</i> <b>in</b> <span style="font-variant: small-caps;">Actions</span>(<i>state</i>) <b>do</b>
<div style="margin-left: 1.5em;"><i>v</i> &larr; <span style="font-variant: small-caps;">Max</span>(<i>v</i>, <span style="font-variant: small-caps;">Min-Value</span>(<span style="font-variant: small-caps;">Result</span>(<i>state</i>, <i>a</i>)))</div>
<b>return</b> <i>v</i>
</div>
<br>
<b>function</b> <span style="font-variant: small-caps; color: #a31515;">Min-Value</span>(<i>state</i>) <b>returns</b> a utility value
<div style="margin-left: 1.5em;">
<b>if</b> <span style="font-variant: small-caps;">Terminal-Test</span>(<i>state</i>) <b>then return</b> <span style="font-variant: small-caps;">Utility</span>(<i>state</i>)<br>
<i>v</i> &larr; +&infin;<br>
<b>for each</b> <i>a</i> <b>in</b> <span style="font-variant: small-caps;">Actions</span>(<i>state</i>) <b>do</b>
<div style="margin-left: 1.5em;"><i>v</i> &larr; <span style="font-variant: small-caps;">Min</span>(<i>v</i>, <span style="font-variant: small-caps;">Max-Value</span>(<span style="font-variant: small-caps;">Result</span>(<i>state</i>, <i>a</i>)))</div>
<b>return</b> <i>v</i>
</div>
</div>

#### An example

First we will cover a small example. From the empty board the tree has hundreds of thousands of states,
so we start from a position in the middle of a game. Only four cells are still free, and it
is `O`'s turn:

In [ ]:
from search.searchtree import draw_search_tree

position = "XX.OOX..."
print_board(position)

<span style="font-size:200%">🫵</span>
It's your turn! Implement `minimax()`. We put `Max-Value` and `Min-Value` into one function:
the player to move decides which of the two applies.

* **Input:** the `game`, and a `state` (a string of nine characters).
* **Returns:** a pair `(value, action)`.
  * `value` is the value of the state: the result of the game if both players play perfectly
    (`+1` if `X` wins, `0` for a draw, `-1` if `O` wins).
  * `action` is the best move for the player to move: the number of the cell to play. If the
    game is already over in this state, there is no move, so return `None`.

Read the comments in the code for more hints.

In [ ]:
def minimax(game, state):
    """
    Returns (value, action) for `state`: its minimax value, and the best action
    for the player to move (None if the game is over in `state`).
    """

    # TODO:
    # - if the state is terminal, return its utility, and None as the action
    # - otherwise, go through all (action, next_state) pairs from
    #   game.successors(state), and get the value of every next_state by
    #   calling minimax() on it
    # - at a MAX state (game.is_max_turn(state)), return the largest value
    #   together with the action that leads to it
    # - at a MIN state, return the smallest value together with its action
    # - call game.successors() only once per state
    pass

#### Verification

`minimax()` returns two things: the **value** of the state, and the **best action** for the
player who is to move. The value tells who wins if both players play perfectly: `+1` for `X`,
`0` for a draw, `-1` for `O`.

Two simple boards to start with. In the first, it is `O`'s turn and `X` threatens to win in
the top row, so `O` has to block in cell 2. The game then ends in a draw. In the second, it is
`X`'s turn, and `X` wins right away by playing cell 2:

In [ ]:
game = TicTacToe()

print_board("XX..O....")
print("your result:", minimax(game, "XX..O...."), "  expected: (0, 2)")

print_board("XX.OO....")
print("your result:", minimax(game, "XX.OO...."), "  expected: (1, 2)")

Each result is `(value, best action)`. It should be the same as the expected one.

Now the example position from above. The expected result is value `0` and best action `2`
again (`O` blocks, then it is a draw). The search has to look at `28` states that are not
finished yet, so `game.expansions` should say `28`. (That works if you call `successors()` only
once per state.)

In [ ]:
game = TicTacToe()
value, action = minimax(game, position)
print("value:", value)
print("best action:", action)
print("states expanded:", game.expansions)

#### Animation

Now watch what **your own** `minimax()` does on this position. Press **▶ Play**, or use
**Step** and the slider.

* Every small board is a state. The top one is the start position, and every level below is
  one more move. The yellow frame shows the state the search is looking at right now.
* The search first goes down until a game is over (grey frame). The number below that board
  is its utility: `+1` if `X` won, `0` for a draw, `-1` if `O` won.
* Then the values travel back up. An orange state (`O` to move, MIN) takes the **smallest**
  value of its moves, and a blue state (`X` to move, MAX) takes the **largest**. While a
  state is not done, it shows the value found so far: `≥ 0` means *MAX gets at least 0*,
  `≤ 0` means *MIN gets at most 0*.
* At the end, the **best line of play** is marked in green. The value at the top is `0`:
  with perfect play this position is a draw, and `O` has to start with cell `2`.

In [ ]:
draw_search_tree(TicTacToe(), position, algorithm=minimax)

### Alpha-beta pruning

Minimax looks at every branch, even when it can already tell that a branch does not
matter. Alpha-beta pruning keeps two numbers while it searches:

* $\alpha$: the best value that MAX can already be sure of, on the way to this state,
* $\beta$: the best value that MIN can already be sure of.

Say we are in a MAX state and one of its moves is already worth at least $\beta$. Then MIN,
higher up in the tree, has a better alternative anyway and will never let the game reach
this state. So the remaining moves cannot change the result, and we can stop looking at them.
In a MIN state it is the same the other way round: once a move is worth at most $\alpha$,
we stop.

The result is exactly the same as for minimax, but with fewer states looked at. The red
boxes show what is new compared to minimax:

<div style="font-family: 'CMU Serif', 'Latin Modern Roman', Georgia, 'Times New Roman', serif; font-size: 1.05em; line-height: 1.6; margin: 1em 0;"><b>function</b> <span style="font-variant: small-caps; color: #a31515;">Max-Value</span>(<i>state</i>, <span style="border: 1px solid red; padding: 0.1em 0.4em; margin-left: -0.4em; border-radius: 3px">&alpha;, &beta;</span>) <b>returns</b> a utility value
<div style="margin-left: 1.5em;">
<b>if</b> <span style="font-variant: small-caps;">Terminal-Test</span>(<i>state</i>) <b>then return</b> <span style="font-variant: small-caps;">Utility</span>(<i>state</i>)<br>
<i>v</i> &larr; &minus;&infin;<br>
<b>for each</b> <i>a</i> <b>in</b> <span style="font-variant: small-caps;">Actions</span>(<i>state</i>) <b>do</b>
<div style="margin-left: 1.5em;"><i>v</i> &larr; <span style="font-variant: small-caps;">Max</span>(<i>v</i>, <span style="font-variant: small-caps;">Min-Value</span>(<span style="font-variant: small-caps;">Result</span>(<i>state</i>, <i>a</i>), <span style="border: 1px solid red; padding: 0.1em 0.4em; margin-left: -0.4em; border-radius: 3px">&alpha;, &beta;</span>))<br>
<span style="border: 1px solid red; padding: 0.1em 0.4em; margin-left: -0.4em; border-radius: 3px"><b>if</b> <i>v</i> &ge; <i>&beta;</i> <b>then return</b> <i>v</i></span><br>
<span style="border: 1px solid red; padding: 0.1em 0.4em; margin-left: -0.4em; border-radius: 3px">&alpha; &larr; <span style="font-variant: small-caps;">Max</span>(&alpha;, <i>v</i>)</span></div>
<b>return</b> <i>v</i>
</div>
<br>
<b>function</b> <span style="font-variant: small-caps; color: #a31515;">Min-Value</span>(<i>state</i>, <span style="border: 1px solid red; padding: 0.1em 0.4em; margin-left: -0.4em; border-radius: 3px">&alpha;, &beta;</span>) <b>returns</b> a utility value
<div style="margin-left: 1.5em;">
<b>if</b> <span style="font-variant: small-caps;">Terminal-Test</span>(<i>state</i>) <b>then return</b> <span style="font-variant: small-caps;">Utility</span>(<i>state</i>)<br>
<i>v</i> &larr; +&infin;<br>
<b>for each</b> <i>a</i> <b>in</b> <span style="font-variant: small-caps;">Actions</span>(<i>state</i>) <b>do</b>
<div style="margin-left: 1.5em;"><i>v</i> &larr; <span style="font-variant: small-caps;">Min</span>(<i>v</i>, <span style="font-variant: small-caps;">Max-Value</span>(<span style="font-variant: small-caps;">Result</span>(<i>state</i>, <i>a</i>), <span style="border: 1px solid red; padding: 0.1em 0.4em; margin-left: -0.4em; border-radius: 3px">&alpha;, &beta;</span>))<br>
<span style="border: 1px solid red; padding: 0.1em 0.4em; margin-left: -0.4em; border-radius: 3px"><b>if</b> <i>v</i> &le; &alpha; <b>then return</b> <i>v</i></span><br>
<span style="border: 1px solid red; padding: 0.1em 0.4em; margin-left: -0.4em; border-radius: 3px">&beta; &larr; <span style="font-variant: small-caps;">Min</span>(&beta;, <i>v</i>)</span></div>
<b>return</b> <i>v</i>
</div>
</div>

<span style="font-size:200%">🫵</span>
It's your turn! Implement `alphabeta()`. Copy your `minimax()` and adapt it.

* **Input:** the same as for `minimax()`, plus `alpha` and `beta`. They start at `-inf` and
  `+inf` and are passed down the tree: `alpha` is the best value that MAX can already be sure
  of, `beta` the best value that MIN can already be sure of.
* **Returns:** the same pair `(value, action)` as `minimax()`. For the root, the value and the
  best action are the same as for minimax, but some branches are skipped on the way.

In [ ]:
def alphabeta(game, state, alpha=-math.inf, beta=math.inf):
    """
    Returns (value, action) for `state`, like minimax(), but stops looking at
    the remaining successors as soon as they cannot change the result.
    """

    # TODO: copy your minimax() from above and adapt it:
    # - pass `alpha` and `beta` on in the recursive calls
    # - at a MAX state, after looking at each successor:
    #   - if the best value so far is >= beta, stop looking at the other
    #     successors, and return the best value and its action right away
    #   - otherwise, set alpha to the larger of alpha and the best value so far
    # - at a MIN state, do the same the other way round: stop if the best value
    #   so far is <= alpha, otherwise set beta to the smaller of beta and the
    #   best value so far
    pass

#### Animation

Now watch **your own** `alphabeta()` search the same position as before. The tree is drawn from
what your function really does. If your function does not skip a state that it should skip, that
state will not be faded out.
Below each state you can see the current values of $\alpha$ and $\beta$. Watch for the red bars:
they mark a cut. Everything below a cut fades out, because the search never looks at it.

In [ ]:
draw_search_tree(TicTacToe(), position, algorithm=alphabeta)

Compare it with the minimax tree. The value at the top and the best line of play should be
the same, but alpha-beta never looks at the faded states. It should expand only `10` of the
`28` unfinished states.

### Playing a game

If both players play perfectly, tic-tac-toe ends in a draw. Let alpha-beta play both sides:

In [ ]:
game = TicTacToe()
state = game.get_start()
while not game.is_terminal(state):
    value, action = alphabeta(game, state)
    state = game.result(state, action)

print_board(state)
if game.utility(state) == 1:
    print("X wins")
elif game.utility(state) == -1:
    print("O wins")
else:
    print("draw")

## From search to learning

Minimax and alpha-beta search need a **complete model** of the game: we know exactly what
every move does, and we assume that the opponent plays perfectly. In the rest of this
notebook, we assume that we do not have such a model. Instead, our program acts on an **environment** and
gets **feedback**. Based on this feedback, it learns how to act. We will teach an elevator
in this way.

### Acting and getting feedback

Think of a loop. The program looks at the current **state** of the environment and chooses an
**action**. The environment reacts and gives back a **reward**, a number that tells how good
or bad that was, and the new state:

```
   state  -->  choose an action  -->  environment  -->  reward + new state  --+
     ^                                                                        |
     +------------------------------------------------------------------------+
```

In the code, an environment is an object with two methods:

| method | what it does |
|---|---|
| `reset()` | starts a new run, called an **episode**, and returns the first state |
| `step(action)` | applies the action and returns `(new state, reward, done)`. `done` is `True` when the episode is over. |

There is a difference to the search exercises. There we could ask for *all* successors of
a state and look ahead. Here we can only take **one** action and see what happens. We cannot
look ahead, and we cannot undo it.

## The problem: an elevator

A single elevator serves a building with five floors. People walk up to a floor and press
▲ or ▼, and the elevator has to carry them to where they want to go.

### How we build the example

A real elevator moves smoothly, and people arrive at any moment. We simplify this so that
learning becomes possible:

* **Space:** we only care about the five floors `F0 … F4`, not about positions in between.
* **Time:** we cut time into discrete steps called **ticks**. In every tick, the elevator chooses
  exactly one action, and then the world moves on by one tick. Nothing happens in between.
* **Episodes:** one run starts with some lamps lit and ends as soon as all lamps are dark (see
  *Episodes* below). No new people arrive during a run.
* **Information:** the elevator only sees the lamps and the car's buttons, not the people.

In [ ]:
import random

from qlearning.elevator import Elevator, ACTIONS, ACTION_NAMES, STOP, CONTINUE, REVERSE
from qlearning.elevator import N_FLOORS, STATES, START_STATES, GAMMA, MAX_STEPS
from qlearning.elevator import GOING_UP, GOING_DOWN, unfinished, step_cost
from qlearning.drawing import draw_state, draw_episode

dark = (0, 0, 0, 0, 0)
elevator = Elevator(seed=0)
state = elevator.reset((0, GOING_UP, dark, (0, 0, 1, 0, 0), dark))
draw_state(state)

This is a cross-section of the building. The elevator car is on floor `F0`. Somebody on
`F2` has pressed ▼, so the lamp next to their landing is lit and a waiting person is drawn
there. The car has its own panel of floor buttons (all dark, because nobody is inside yet).
The little arrow on that panel shows which way the car is heading.

### What the elevator can see

In a lobby there are just two call buttons, ▲ and ▼. A lit lamp means *somebody here wants
to go that way*, and that is all the elevator knows. Both lamps can be lit at the same
time: some people waiting to go up, others waiting to go down.

Inside the car there is a row of floor buttons. A passenger presses theirs only after
getting in, so the elevator learns where somebody is going **after** it has picked them up.

We describe what the elevator can see: which lamps are lit.
If two people on `F2` both press ▼, the lamp is simply lit. The elevator cannot tell one
person from two.

### The state

A state is `(floor, direction, up, down, car)`:

| part | meaning |
|---|---|
| `floor` | where the cabin is, `0…4` |
| `direction` | which way the car/cabin is travelling: `0` up, `1` down |
| `up`, `down` | one flag per floor: is the ▲ / ▼ lamp on that landing lit? defined as python `tuples` |
| `car` | one flag per floor: is that button lit on the cabin's panel? defined as a python `tuple` |

Here is the state we just drew:

In [ ]:
print(state)
print(len(state))

* `floor` is `0` and `direction` is `0`: the car is on the ground floor, heading up.
* `down` is `(0, 0, 1, 0, 0)`: the ▼ lamp on `F2` is lit. No ▲ lamp is lit (`up`).
* `car` is all zeros: no button is lit on the car's panel.

How many different states are there?

| part | options | why |
|---|---|---|
| `floor` and `direction` | `8` | 5 floors with 2 directions each would give 10, but the car cannot head down on `F0` or up on `F4`. It turns around there by itself (more on that below). |
| `up` | `16` | 4 floors have a ▲ button (the top floor has none), each lit or dark: 2⁴ |
| `down` | `16` | 4 floors have a ▼ button (the ground floor has none), each lit or dark: 2⁴ |
| `car` | `32` | 5 buttons, each lit or dark: 2⁵ |

Together that gives `8` × `16` × `16` × `32` = **65 536 states**.

### Direction of travel

The car does not wander around. It always has a direction. It keeps going that way, picks
up everybody who wants to go the same way, and only then turns around. Say the car is
heading up to `F4` and you are on `F1` pressing ▼: the car will **not** automatically come back for you
right away. You have to wait until it returns or decides to reverse the direction.

In every tick, the elevator chooses one of three actions:

| id | name | effect |
|----|------|--------|
| `0` | `Stop` | Open the doors. People whose floor this is get out. People waiting here who want to go the same way as the car get in. |
| `1` | `Continue` | Move one floor in the current direction. |
| `2` | `Reverse` | Turn the car around. This takes a tick, and the car does not move. |

Two things follow from this:

* While going up, `Stop` only picks up people who pressed ▲. The ▼ lamp on that floor
  stays lit until the car comes back down as this behavior reflects the real-world.
* There is no "go down" action. To go the other way, the elevator has to spend a tick on
  `Reverse`.

One more rule: **the car turns around by itself at the top and the bottom.** It cannot head
down from `F0` or up from `F4`, so when it arrives there, the direction flips
automatically. That is why `Reverse` on the ground floor does nothing useful but still
costs a tick.

**Note** that we came up with this simplified scenario to keep environment complexity manageable. You may come up with a more realistic encoding.

Let us try it. We drive up to `F2`, where the ▼ lamp is lit, and stop there anyway:

In [ ]:
state, reward, done = elevator.step(CONTINUE)
state, reward, done = elevator.step(CONTINUE)
print("at F2, sweeping up:", state)
state, reward, done = elevator.step(STOP)
print("after stopping:    ", state, "   <- the ▼ lamp is still lit!")

Nothing happened: the lamp is still lit, and we wasted a tick with the doors open. The car
is heading up, but this passenger wants to go down. To pick them up, the car has to
`Reverse` first:

In [ ]:
for action in [REVERSE, STOP]:
    state, reward, done = elevator.step(action)
    print(ACTION_NAMES[action], "->", state, "reward", reward)

Now the ▼ lamp is out and a button has lit up on the car's panel.

### The reward

In every tick, the elevator receives a **reward**. We define it as the negative number of lit
lamps. A lit lamp is a person waiting for the car, or a button lit on the car's panel.
With the flags $up_f$, $down_f$ and $car_f$ from the state (`1` if the lamp is lit, `0` if
it is dark), the reward in a state is

$$r \;=\; -\Big(\sum_{f=0}^{4} up_f \;+\; \sum_{f=0}^{4} down_f \;+\; \sum_{f=0}^{4} car_f\Big)$$

Waiting around is expensive: the more lamps are lit, the more negative the reward. The
function `step_cost()` below computes the cost, which is the negative of the reward, and it
is what the demo above printed after each step.

Over a whole episode, the rewards of all ticks add up. The elevator wants this sum to be as
large as possible, so close to $0$.

If you are interested in the implementation details, look at `qlearning/elevator.py`.

In [ ]:
busy = (2, GOING_UP, (0, 1, 0, 0, 0), (0, 0, 0, 1, 0), (1, 0, 0, 0, 1))   # four lamps lit
draw_state(busy)
print("cost of one tick:", step_cost(busy, CONTINUE))

Four lamps are lit, so this tick has reward $-4$, no matter which action we choose. The reward
only depends on how many lamps are lit, so the only way to pay less is to get the lamps dark
sooner. Maximising the reward therefore means getting everybody where they are going as
quickly as possible.

That is all we tell the elevator. We never say *how* to behave, only what it costs when it
behaves badly.

### Episodes

One run of the elevator is called an **episode**. It works like this:

1. `reset()` creates a start state in which at least one lamp is lit.
2. In every tick, the elevator chooses an action and calls `step(action)`. It gets back the
   next state and the reward.
3. The episode is **over** when `step()` returns `done=True`, which happens as soon as all
   lamps are dark: everybody has been picked up and delivered.
4. As a safety net, we also stop after `MAX_STEPS = 200` ticks, in case an elevator drives
   around without ever finishing the job.

The **total reward** of an episode is the sum of the rewards of all its ticks. A good elevator
finishes quickly, and so its total reward is close to $0$.

### Controllers

A **controller** is a function that receives the current state and returns the action to
take. (In the lecture this is called a *policy*.) The simplest one is *random control*: it
ignores the state completely and picks an action at random.

In [ ]:
def random_control(state) -> int:
    return random.choice(ACTIONS)

To compare controllers fairly, we need two general tools. They are already written for you
in `qlearning/controllers.py`:

* `run_episode(controller, env, start)` runs one episode and returns the states visited and
  the actions taken,
* `average_cost(controller)` runs many episodes from random situations and returns the
  average cost, and how often the elevator cleared the building.

In [ ]:
from qlearning.controllers import run_episode, average_cost

In [ ]:
random.seed(0)
cost, cleared = average_cost(random_control)
print("random control: cost per episode", round(cost, 1), ", cleared", round(100 * cleared), "%")

That is the number to beat: random control costs around 300 per episode, and does not even
finish the job every time. Can a controller that has never heard of elevators learn to do
better?


Here is a situation we will use for the checks and the demos below:

In [ ]:
situation = (0, GOING_UP, (0, 1, 0, 0, 0), (0, 0, 0, 1, 0), (0, 0, 0, 0, 1))
draw_state(situation)

## Q-learning

The idea from the lecture: for every state $s$ and every action $a$, we keep a number
$Q(s,a)$. It answers the question *"If I take action $a$ in state $s$, how much reward will
I collect from now on, assuming I act sensibly afterwards?"* Here every reward is negative,
so every $Q$ is negative too, and a value closer to $0$ is better.

If we knew all these numbers, acting would be easy: in each state, take the action with the
highest $Q$.

We do not know them. But we can start with a guess and correct it a little every time the
elevator tries something and sees what happens. Random control never does that: it learns
nothing from what happened.

### The Q-table

Our elevator has 65 536 states, which is few enough to store the Q-values in a table: one
row for every state, with three numbers in it, one for each action. Let us check the sizes:

In [ ]:
print(f'{len(STATES):,} states x {len(ACTIONS)} actions')

That is the same 65 536 states we counted earlier.

We start every entry at `-100`. This is a deliberately **pessimistic** guess, and it
matters. Every real value is negative, because every tick costs something. If we started at
`0`, an action we have never tried would look better than anything we have actually
measured, so a greedy controller would keep picking actions it knows nothing about.

Starting at `-100` means the opposite: an action we have tried and measured looks better
than one we have not.

<span style="font-size:200%">🫵</span>

It's your turn! Build the Q-table.

* **Input:** the number `initial` that every entry should start with.
* **Returns:** a dictionary with one entry for each state in `STATES`. Each entry is a list
  with one number per action (`len(ACTIONS)` numbers), all set to `initial`.

Hint: You can use the variables `STATES` and `ACTIONS`.

In [ ]:
def make_q_table(initial=-100.0):
    """
    Returns the Q-table: a dict that maps every state in STATES to a list of
    len(ACTIONS) values, one per action, all equal to `initial`.
    """

    # TODO: return a dictionary mapping every state in STATES to a list of
    # len(ACTIONS) values, each set to `initial`
    pass

In [ ]:
Q = make_q_table()
print("entries:", len(Q))
print("one row:", Q[situation])

It should print 65 536 entries and a row of three `-100.0` values.

### Acting: $\varepsilon$-greedy

While learning, the elevator cannot always take the action it currently thinks is best.
Then it would never find out whether something else is better. So it does two things:

* with probability $\varepsilon$ it takes a random action (**explore**),
* otherwise it takes the action with the highest Q-value (**exploit**).

<div style="font-family: 'CMU Serif', 'Latin Modern Roman', Georgia, 'Times New Roman', serif; font-size: 1.05em; line-height: 1.6; margin: 1em 0;"><b>function</b> <span style="font-variant: small-caps; color: #a31515;">Epsilon-Greedy</span>(<i>Q</i>, <i>state</i>, <i>&epsilon;</i>) <b>returns</b> an action
<div style="margin-left: 1.5em;"><b>with probability</b> <i>&epsilon;</i> <b>return</b> a uniformly random action<br>
<b>otherwise return</b> arg max<sub><i>a</i></sub> <i>Q</i>[<i>state</i>][<i>a</i>]
</div>
</div>

<span style="font-size:200%">🫵</span>

Implement ε-greedy.

* **Input:** the `q_table`, a `state`, and `epsilon`, the probability of exploring.
* **Returns:** one action, a number from `ACTIONS`. It does not change the table.

One detail: at the very beginning, all three values in a row are equal. If you always picked the first of the best actions, the elevator would just keep opening its doors. So when several actions share the highest value, pick one of them **at random**.

In [ ]:
def epsilon_greedy(q_table, state, epsilon):
    """
    Returns the action to take in `state`: a random one with probability
    `epsilon`, otherwise one with the highest Q-value (ties broken at random).
    """

    # TODO:
    # - with probability `epsilon`, return a random action from ACTIONS
    # - otherwise, look at the row q_table[state] and return an action with
    #   the highest value
    # - if several actions share the highest value, pick among them at random
    #   (hint: build the list of best actions first, then random.choice() it)
    pass

With `epsilon=1` it should ignore the table completely and choose all three actions about equally often:

In [ ]:
random.seed(0)
Q = make_q_table()
counts = [0, 0, 0]
for _ in range(3000):
    counts[epsilon_greedy(Q, situation, epsilon=1.0)] += 1
print("epsilon=1.0 :", counts)

And with `epsilon=0` it should always choose the best action according to the table:

In [ ]:
Q[situation] = [-5.0, -9.0, -7.0]       # pretend we learned something
print("epsilon=0.0 :", ACTION_NAMES[epsilon_greedy(Q, situation, epsilon=0.0)])

That should print `Stop`, because `-5.0` is the largest of the three values.

### Learning: the update

Now the actual algorithm. The elevator took action $a$ in state $s$, received reward $r$,
and ended up in state $s'$. What does this tell us about $Q(s,a)$?

The value of taking $a$ in $s$ should be roughly the reward $r$ we just got, plus whatever
we can still collect from $s'$. From $s'$ we would take the best action we know, so:

$$\text{target} \;=\; r + \gamma \max_{a'} Q(s',a')$$

$\gamma = 0.99$ shrinks the future a little, so a reward now counts slightly more than the
same reward later. If the episode just ended, there is no future left, and the target is
simply $r$.

We do **not** replace $Q(s,a)$ by the target. We move it a small step towards it, a
fraction $\alpha$ of the way (the *learning rate*):

<div style="font-family: 'CMU Serif', 'Latin Modern Roman', Georgia, 'Times New Roman', serif; font-size: 1.05em; line-height: 1.6; margin: 1em 0;"><b>function</b> <span style="font-variant: small-caps; color: #a31515;">Q-Update</span>(<i>Q</i>, <i>s</i>, <i>a</i>, <i>r</i>, <i>s&prime;</i>, <i>done</i>) 
<div style="margin-left: 1.5em;"><b>if</b> <i>done</i> <b>then</b> <i>target</i> &larr; <i>r</i><br>
<b>else</b> <i>target</i> &larr; <i>r</i> + <i>&gamma;</i> &middot; max<sub><i>a&prime;</i></sub> <i>Q</i>[<i>s&prime;</i>][<i>a&prime;</i>]<br>
<i>Q</i>[<i>s</i>][<i>a</i>] &larr; <i>Q</i>[<i>s</i>][<i>a</i>] + <i>&alpha;</i> &middot; (<i>target</i> &minus; <i>Q</i>[<i>s</i>][<i>a</i>])
</div>
</div>

Why only a small step? Because the target is **noisy**. When the elevator stops for a ▲
lamp, the new passenger can press any floor above. So the same state and action lead to
different next states on different visits, and therefore to different targets. With
$\alpha = 1$ we would throw away everything we knew and trust whichever outcome came last.
A smaller $\alpha$ averages over many visits instead.

<span style="font-size:200%">🫵</span>

Implement the update.

* **Input:** the `q_table`, and what just happened: in `state` the elevator took `action`, got
  `reward`, and ended up in `next_state`. `done` says whether the episode ended with this step.
  `alpha` is the learning rate.
* **Returns:** nothing. It changes `q_table[state][action]` in place.

In [ ]:
def q_learning_update(q_table, state, action, reward, next_state, done, alpha, gamma=GAMMA):
    """
    Moves q_table[state][action] a fraction `alpha` towards the target
    (reward, plus the discounted best value of next_state). Returns nothing.
    """

    # TODO:
    # - if `done`, the target is just `reward`
    # - otherwise it is reward + gamma * (the largest value in q_table[next_state])
    # - then move q_table[state][action] a fraction `alpha` towards the target:
    #       old + alpha * (target - old)
    pass

A quick check. Suppose the episode ends after this step (`done=True`) with reward `-2`.
Then the target is just `-2`, and with `alpha=0.5` an entry at `-100` should move halfway
there, to `-51`:

In [ ]:
Q = make_q_table()
q_learning_update(Q, situation, STOP, reward=-2.0, next_state=situation, done=True, alpha=0.5)
print(Q[situation])

### Putting it together

One episode of learning: start a new episode, and then in every tick pick an action with
$\varepsilon$-greedy, apply it, and update the table with what happened.

<span style="font-size:200%">🫵</span>

The function below plays one episode with random actions and learns nothing. Turn it into Q-learning by making the two changes marked in the code.

* **Input:** the `q_table`, an environment `env`, the exploration rate `epsilon` and the learning rate `alpha`.
* **Returns:** the total reward collected in the episode (a negative number). The Q-table is changed in place.

In [ ]:
def learn_episode(q_table, env, epsilon, alpha):
    """
    Plays one episode with epsilon-greedy actions and updates `q_table` after
    every step. Returns the total reward of the episode.
    """

    # TODO: change the two marked lines.
    # - choose the action with epsilon_greedy() instead of at random
    # - after every step, call q_learning_update() with what just happened
    # Everything else can stay as it is.

    state = env.reset()
    total = 0.0
    for _ in range(MAX_STEPS):
        action = random.choice(ACTIONS)                       # <- change this
        next_state, reward, done = env.step(action)
        # <- and add the update here
        state = next_state
        total += reward
        if done:
            break
    return total

### Training

The last piece is given. `train()` calls `learn_episode()` many times. It lowers
$\varepsilon$ slowly from 1 to 0.05: at first the elevator mostly explores, later it mostly
uses what it has learned.

For this scenario, each episode starts in a state chosen uniformly at random, in order to encourage a broad coverage of states visited. This is called *exploring starts*.

In [ ]:
from tqdm.auto import tqdm

def train(q_table: dict, episodes: int = 1_000_000, alpha: float = 0.2,
          seed: int = 0) -> list[float]:
    """
    Trains the table in place and returns the total reward of every episode.
    """
    env = Elevator(seed=seed)
    random.seed(seed)
    history = []
    progress = tqdm(range(episodes), desc="training", unit="ep",
                    mininterval=0.5, miniters=2000)
    for episode in progress:
        epsilon = max(0.05, 1.0 - episode / (0.8 * episodes))
        history.append(learn_episode(q_table, env, epsilon, alpha))
        if (episode + 1) % 20_000 == 0:
            progress.set_postfix(reward=f"{sum(history[-20_000:]) / 20_000:.1f}",
                                 eps=f"{epsilon:.2f}")
    progress.close()
    return history

This takes **about a minute or two**. Watch the average reward on the progress bar: it starts very low (random control scores about -300) and climbs as the table fills in.

In [ ]:
Q = make_q_table()
history = train(Q)

In [ ]:
from qlearning.drawing import plot_learning_curve

plot_learning_curve(history)

### Verification

With the table trained, the elevator no longer needs to explore. It simply takes the best
action it knows, which is $\varepsilon$-greedy with $\varepsilon = 0$:

In [ ]:
def learned_control(state, q_table) -> int:
    return epsilon_greedy(q_table, state, epsilon=0.0)

Let us give it the situation from above: three lamps are lit, and the car is at the bottom.

In [ ]:
states, actions = run_episode(lambda x: learned_control(x, Q), Elevator(seed=0), situation)
print(f"{len(actions)} ticks:", ", ".join(ACTION_NAMES[a] for a in actions))
draw_episode(states, actions, q_table=Q)

Press **▶ Play**. The little arrow on the car's panel shows which way the car is heading.
It turns grey while the doors are open. Whenever the car changes direction somewhere in the
middle of the shaft, it has chosen `Reverse`, which cost a tick.

The bars on the right are the three Q-values of the state on the screen, so you can see
*why* the elevator did what it did. The shortest bar is the best action, because all values
are negative.

### How good is it?

We compare two controllers on the same 1000 random situations: the one we trained, and
random control.

In [ ]:
random.seed(0)
for name, controller in [("learned", lambda x: learned_control(x, Q)), ("random", random_control)]:
    cost, cleared = average_cost(controller)
    print(name, ": cost per episode", round(cost, 1), ", cleared", round(100 * cleared), "%")

Lower cost is better. The learned controller needs only about a sixth of the cost of random
control, and it clears the building every time.

Nobody told the elevator what an elevator is, that `Reverse` on the ground floor is
pointless, or that the doors should open where people are waiting. It worked all of that out
from one number per tick: what the last move cost. The Q-values are where it keeps this
knowledge, one row for every state.

The learning curve above shows the other half of the story: the reward rises as the table
fills in.

### Playground

Now it is your turn to drive. Press the call buttons and watch how the car reacts. You can
switch between the learned controller and random control while it runs.

In [ ]:
from qlearning.playground import Playground

Playground(Q, {'learned': lambda x: learned_control(x, Q),
               'random': random_control}).show()